In [1]:
from data_loader import get_mix_instruct
from utility_functions.delift_se import get_delift_se_utility
from utility_functions.encodes import get_encodes_utility
from subset import create_subset, get_subset

prompts, references, ds_name = get_mix_instruct("train", 21000)
utility, utility_name = get_delift_se_utility(prompts, references, ds_name)
utility_enc, utility_name_enc = get_encodes_utility(prompts, references, ds_name)
subset, subset_name = create_subset(utility, utility_name, k =1)
subset_enc, subset_name_enc = create_subset(utility_enc, utility_name_enc, k =1)
s_prompts, s_references = get_subset(subset, prompts, references)

prompts_val, references_val, _ = get_mix_instruct("validation", 50)

Dataset: mix-instruct_train_21000 found in cache, loading from cache ✅
Utility: mix-instruct_train_21000_delift-se found in cache, loading from cache ✅
Utility: mix-instruct_train_21000_encodes found in cache, loading from cache ✅
Subset: mix-instruct_train_21000_delift-se_1 found in cache, loading from cache ✅
Subset: mix-instruct_train_21000_encodes_1 found in cache, loading from cache ✅
Dataset: mix-instruct_validation_50 found in cache, loading from cache ✅


In [6]:
from scipy.stats import spearmanr
import numpy as np

x = np.array(subset_enc)[:, 0].astype(int)
y = x

correlation, p_value = spearmanr(x, y)
print(f"Spearman's Rank Correlation: {correlation}")

Spearman's Rank Correlation: 1.0


In [3]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("BAAI/bge-base-en")

2025-03-30 22:04:24.333368: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-30 22:04:24.349434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743372264.367031   21458 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743372264.372303   21458 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743372264.386097   21458 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [12]:
sentences = ["I love AI.", "AI."]
embeddings = model.encode(sentences)
similarity = util.cos_sim(embeddings[0], embeddings[1])

print("Cosine Similarity:", similarity.item())

Cosine Similarity: 0.8825727701187134


In [4]:
train_samples = [
    InputExample(texts=[f'{prompt1} {reference1}', f'{prompt2} {reference2}'], label=0.0)
    for prompt1, reference1 in zip(prompts_val, references_val) for prompt2, reference2 in zip(prompts_val, references_val)
]
len(train_samples)

2500

In [ ]:
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=32)
train_loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="cache/bge-finetuned"
)
